In [2]:
import pandas as pd 
import numpy as np 
import torch 

In [15]:
train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')


In [15]:


train_df['xrd_peak_locations'] = train_df['xrd_peak_locations'].apply(
    lambda x: [float(i) for i in x.strip('[]').split(',')]
)

#make sure we can read in the diffraction patterns
train_df['xrd_peak_intensities'] = train_df['xrd_peak_intensities'].apply(
    lambda x: [float(i) for i in x.strip('[]').split(',')]
)


In [3]:

from tqdm import tqdm

import numpy as np

def pseudo_voigt(x, center, amplitude, fwhm, eta):
    """
    Pseudo-Voigt function.
    x: array-like, the independent variable
    center: float, the center of the peak
    amplitude: float, the height of the peak
    fwhm: float, full-width at half-maximum
    eta: float, the fraction of the Lorentzian component (0 <= eta <= 1)
    """
    sigma = fwhm / (2 * np.sqrt(2 * np.log(2)))  # Convert FWHM to sigma for Gaussian
    lorentzian = amplitude * (fwhm**2 / ((x - center)**2 + fwhm**2))
    gaussian = amplitude * np.exp(-(x - center)**2 / (2 * sigma**2))
    return eta * lorentzian + (1 - eta) * gaussian

def superimposed_pseudo_voigt(x, locations, intensities, fwhm, eta):
    """
    Superimpose multiple pseudo-Voigt functions.
    x: array-like, the independent variable
    locations: list, the centers of the peaks
    intensities: list, the heights of the peaks
    fwhm: float, full-width at half-maximum (same for all peaks)
    eta: float, the fraction of the Lorentzian component (0 <= eta <= 1, same for all peaks)
    """
    total = np.zeros_like(x)
    for center, amplitude in zip(locations, intensities):
        total += pseudo_voigt(x, center, amplitude, fwhm, eta)
    total = total / max(total)
    return total

import pandas as pd
from multiprocessing import Pool


# Assuming train_df is already defined and simulate_xrd function is defined

# Function to simulate XRD for each row
def simulate_pv_xrd_for_row(row_tuple):
    index, row = row_tuple  # Unpack the tuple
    
    x = np.linspace(5, 85, 8192) 
    fwhm = 0.005  # Full-width at half-maximum (common for all peaks)
    eta = 0.75  # Fraction of Lorentzian component (common for all peaks)

    sim_xrd = superimposed_pseudo_voigt(x, row['xrd_peak_locations'], row['xrd_peak_intensities'], fwhm, eta)

    return sim_xrd

# Function to apply simulation to each row
def apply_simulation(data):
    with Pool() as pool:
        results = list(tqdm(pool.imap(simulate_pv_xrd_for_row, data.iterrows()), total=len(data)))
    return results

In [18]:
# Apply the simulation
train_df['sim_pv_xrd_intensities'] = pd.Series(apply_simulation(train_df), dtype=object)


100%|██████████| 27136/27136 [06:23<00:00, 70.79it/s] 


In [4]:
def convert_to_ast_lit_eval(data):

    data['xrd_peak_locations'] = data['xrd_peak_locations'].apply(
        lambda x: [float(i) for i in x.strip('[]').split(',')]
    )

    #make sure we can read in the diffraction patterns
    data['xrd_peak_intensities'] = data['xrd_peak_intensities'].apply(
        lambda x: [float(i) for i in x.strip('[]').split(',')]
    )

    return data

In [5]:
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')

In [6]:
#repeat for test and val

#convert test and val to ast lit eval
test_df = convert_to_ast_lit_eval(test_df)
val_df = convert_to_ast_lit_eval(val_df)


In [7]:
test_df['sim_pv_xrd_intensities'] = pd.Series(apply_simulation(test_df), dtype=object)

100%|██████████| 9047/9047 [02:10<00:00, 69.16it/s] 


In [8]:
val_df['sim_pv_xrd_intensities'] = pd.Series(apply_simulation(val_df), dtype=object)

100%|██████████| 4382/4382 [01:02<00:00, 70.29it/s] 


In [9]:
def stack_and_save(data, filename):
    data_xrd = np.array(data['sim_pv_xrd_intensities'])
    data = np.stack(data_xrd)
    data = data.reshape(data.shape[0], 8192, 1)
    np.save(filename, data)
    print("Saved to {}".format(filename))

In [21]:
#save the data
stack_and_save(train_df, '/home/gridsan/tmackey/cdvae/data/mp_20/train_pv_xrd.npy')

Saved to /home/gridsan/tmackey/cdvae/data/mp_20/train_pv_xrd.npy


KeyError: 'sim_pv_xrd_intensities'

In [10]:
stack_and_save(test_df, '/home/gridsan/tmackey/cdvae/data/mp_20/test_pv_xrd.npy')
stack_and_save(val_df, '/home/gridsan/tmackey/cdvae/data/mp_20/val_pv_xrd.npy')

Saved to /home/gridsan/tmackey/cdvae/data/mp_20/test_pv_xrd.npy
Saved to /home/gridsan/tmackey/cdvae/data/mp_20/val_pv_xrd.npy


In [1]:
import pandas as pd 
import numpy as np 
import torch 

In [2]:
#load the data back in to double check 
train = np.load('/home/gridsan/tmackey/cdvae/data/mp_20/train_pv_xrd.npy')
test = np.load('/home/gridsan/tmackey/cdvae/data/mp_20/test_pv_xrd.npy')
val = np.load('/home/gridsan/tmackey/cdvae/data/mp_20/val_pv_xrd.npy')

In [6]:
train_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/train.csv')
test_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/test.csv')
val_df = pd.read_csv('/home/gridsan/tmackey/cdvae/data/mp_20/val.csv')

In [3]:
#convert the data to torch tensors
train = torch.from_numpy(train).float()
test = torch.from_numpy(test).float()
val = torch.from_numpy(val).float()

In [4]:
#reshape
train = train.reshape(train.shape[0], train.shape[2], train.shape[1])
test = test.reshape(test.shape[0], test.shape[2], test.shape[1])
val = val.reshape(val.shape[0], val.shape[2], val.shape[1])


In [7]:
#make a train dict with the entries in the material_id column as keys and the corresponding tensor as values
train_dict = {}
for i in range(len(train_df)):
    train_dict[train_df['material_id'][i]] = train[i]
    


In [8]:
#repeat for test and val
test_dict = {}
for i in range(len(test_df)):
    test_dict[test_df['material_id'][i]] = test[i]

val_dict = {}

for i in range(len(val_df)):
    val_dict[val_df['material_id'][i]] = val[i]

In [11]:
#save the dicts to pt files
torch.save(train_dict, '/home/gridsan/tmackey/cdvae/data/mp_20/train_pv_xrd.pt')
torch.save(test_dict, '/home/gridsan/tmackey/cdvae/data/mp_20/test_pv_xrd.pt')
torch.save(val_dict, '/home/gridsan/tmackey/cdvae/data/mp_20/val_pv_xrd.pt')

In [12]:
val_dict[ 'mp-754553'].shape

torch.Size([1, 8192])

In [28]:
xrd_peak_intensities_dict = torch.load('/home/gridsan/tmackey/cdvae/data/mp_20_dm/train' + "_xrd_peak_intensities_dict.pt")

In [31]:
xrd_peak_intensities_dict['mp-1221227'].shape

torch.Size([1, 256])